In [1]:
# Fetching Dataset, the tiny Dataset shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-07-06 22:17:00--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.006s  

2026-07-06 22:17:00 (178 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# Read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [4]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
 # let's now encode the entire text dataset and store it into a torch.tensor
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

In [7]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

# Characters and vocab defined in earlier cells
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [8]:
# Verify the model connection using the global config and GPT class
try:
    # Use the global config defined in be0e106f
    model_verify = GPT(config).to(device)

    # Prepare a tiny batch for verification
    xb_tiny, yb_tiny = get_batch('train')
    logits_tiny, loss_tiny = model_verify(xb_tiny, yb_tiny)

    print(f"Success! Model initialized with {sum(p.numel() for p in model_verify.parameters())/1e6:.2f}M params.")
    print(f"Logits shape: {logits_tiny.shape}")
    print(f"Initial loss: {loss_tiny.item():.4f}")
except NameError as e:
    print(f"Verification failed: {e}. Ensure cells i4v2Sj-te5Lg, 70VN7w2ofQFW, and be0e106f have been executed.")

Verification failed: name 'GPT' is not defined. Ensure cells i4v2Sj-te5Lg, 70VN7w2ofQFW, and be0e106f have been executed.


In [9]:
import torch
from dataclasses import dataclass

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True

# Define the core Model Configuration needed for get_batch
# Assuming vocab_size is already defined from previous cells (e.g., qb1cIrz_PTmv)
config = GPTConfig(
    block_size=256,
    vocab_size=vocab_size,
    n_layer=6,
    n_head=6,
    n_embd=384,
    dropout=0.2
)

# Define Training-specific Hyperparameters needed for get_batch
batch_size = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Split up the data into train and validation sets
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - config.block_size, (batch_size,))
    x = torch.stack([data_split[i:i+config.block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+config.block_size+1] for i in ix])
    # Move to the defined device
    x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch('train')
print(f'Train batch x shape: {xb.shape}, y shape: {yb.shape}, device: {xb.device}')

Train batch x shape: torch.Size([64, 256]), y shape: torch.Size([64, 256]), device: cuda:0


In [10]:
# Simple training loop to verify everything works
import torch
import torch.nn as nn
from torch.nn import functional as F

# --- Required Class Definitions ---

class Head(nn.Module):
    def __init__(self, config, head_size):
        super().__init__()
        self.key = nn.Linear(config.n_embd, head_size, bias=False)
        self.query = nn.Linear(config.n_embd, head_size, bias=False)
        self.value = nn.Linear(config.n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(config.block_size, config.block_size)))
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, config, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(config, head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.ReLU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        head_size = config.n_embd // config.n_head
        self.sa = MultiHeadAttention(config, config.n_head, head_size)
        self.ffwd = FeedFoward(config)
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None: torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h: x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# --- Execution ---

model_verify = GPT(config).to(device)
optimizer = torch.optim.AdamW(model_verify.parameters(), lr=3e-4, weight_decay=0.1)

model_verify.train()
for iter in range(10):
    xb, yb = get_batch('train')
    logits, loss = model_verify(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if iter % 2 == 0: print(f"Step {iter}: loss {loss.item():.4f}")

Step 0: loss 4.2049
Step 2: loss 3.5560
Step 4: loss 3.4377
Step 6: loss 3.3460
Step 8: loss 3.3348


In [11]:
# Finally, let's generate some text from our (very) lightly trained model
# Ensure the model and context are on the same device
model_verify.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device) # Move context to 'cuda'
generated_chars = model_verify.generate(context, max_new_tokens=100)[0].tolist()
print(decode(generated_chars))


e ei iimsi dthsyhapscCr sohJ
hreceterheys sWlq: & s an.y 
w
  aaIeXsbeh aeqofann.s q
  e g  t  rreb 


In [12]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True

In [13]:
import matplotlib.pyplot as plt

# 1. Define the core Model Configuration first
config = GPTConfig(
    block_size=256,
    vocab_size=vocab_size,
    n_layer=6,
    n_head=6,
    n_embd=384,
    dropout=0.2
)

# 2. Define Training-specific Hyperparameters
batch_size = 64
max_iters = 3000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

# 3. Create global aliases from the config for the rest of the script to use
# This ensures that if you change 'config.n_embd', the rest of the code follows.
block_size = config.block_size
n_embd = config.n_embd
n_head = config.n_head
dropout = config.dropout

# Initialize results dictionary
if 'results' not in globals():
    results = {
        'with_attention': {'train': [], 'val': [], 'steps': []},
        'no_attention': {'train': [], 'val': [], 'steps': []}
    }

### Detailed Breakdown of the Transformer Architecture

1.  **Head (Self-Attention)**:
    *   `key`, `query`, `value`: Linear projections of the input.
    *   `wei = q @ k.transpose`: Calculates the 'affinity' between tokens.
    *   `masked_fill`: Ensures the decoder cannot look into the future (causality).
    *   `softmax`: Normalizes weights so they sum to 1.
2.  **MultiHeadAttention**: Runs multiple `Head` instances in parallel, allowing the model to attend to different parts of the sequence for different reasons (e.g., one head for grammar, one for context).
3.  **FeedForward**: A point-wise sub-layer that allows tokens to 'think' individually about the information gathered during attention.
4.  **Block**: Combines Attention and FeedForward with **Residual Connections** (`x = x + ...`) and **Layer Normalization** to ensure stable gradients during deep training.

In [ ]:
model = GPT(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"Training custom GPT with Multi-Head Attention ({sum(p.numel() for p in model.parameters())/1e6:.2f}M params)... ")

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        results['with_attention']['train'].append(losses['train'].item())
        results['with_attention']['val'].append(losses['val'].item())
        results['with_attention']['steps'].append(iter)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Training custom GPT with Multi-Head Attention (10.76M params)... 
step 0: train loss 4.1674, val loss 4.1641
step 500: train loss 1.9092, val loss 2.0194
step 1000: train loss 1.4939, val loss 1.6919


In [ ]:
class NoAttentionBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        # FeedFoward now expects the config object, not just an integer
        self.ffwd = FeedFoward(config)
        self.ln1 = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        # Skip the self-attention layer entirely
        x = x + self.ffwd(self.ln1(x))
        return x

# Create a GPT variant without attention heads
model_no_attn = GPT(config).to(device)
# Manually swap blocks to NoAttention versions using the config object
model_no_attn.transformer.h = nn.ModuleList([NoAttentionBlock(config) for _ in range(config.n_layer)]).to(device)

optimizer = torch.optim.AdamW(model_no_attn.parameters(), lr=learning_rate)

print(f"Training GPT WITHOUT Attention ({sum(p.numel() for p in model_no_attn.parameters())/1e6:.2f}M params)... ")

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        # Modified estimate_loss for the no_attn model
        model_no_attn.eval()
        out = {}
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                X, Y = X.to(device), Y.to(device)
                _, loss = model_no_attn(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
        model_no_attn.train()

        results['no_attention']['train'].append(out['train'].item())
        results['no_attention']['val'].append(out['val'].item())
        results['no_attention']['steps'].append(iter)
        print(f"step {iter}: train loss {out['train']:.4f}, val loss {out['val']:.4f}")

    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)
    logits, loss = model_no_attn(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
plt.figure(figsize=(12, 7))
# With Attention
plt.plot(results['with_attention']['steps'], results['with_attention']['train'], 'b-', label='With Attention (Train)')
plt.plot(results['with_attention']['steps'], results['with_attention']['val'], 'b--', label='With Attention (Val)')
# No Attention
plt.plot(results['no_attention']['steps'], results['no_attention']['train'], 'r-', label='No Attention (Train)')
plt.plot(results['no_attention']['steps'], results['no_attention']['val'], 'r--', label='No Attention (Val)')

plt.title('Impact of Attention Mechanism on Training')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

### Line-by-Line Documentation

- `Head`: Implements one channel of self-attention. It projects inputs into Queries, Keys, and Values. The dot-product of Q and K (scaled by square root of dimension) creates the attention matrix, which is then masked to prevent looking ahead.
- `MultiHeadAttention`: Wraps multiple `Head` instances and concatenates their outputs, followed by a linear projection. This allows the model to learn multiple relationships for each token.
- `FeedForward`: A standard MLP applied to each token position independently.
- `Block`: The fundamental building block. It uses LayerNorm before the sub-layers (Pre-Norm architecture) and residual connections to keep the signal strong.
- `GPT`: The top-level container. It handles token and position embeddings, stacks the Blocks, and applies the final LayerNorm and Linear head to produce vocabulary logits.

In [ ]:
# Re-train the baseline model (No Attention) for a clean comparison
model_no_attn = GPT(config).to(device)
# Fixed: Pass 'config' instead of individual members to match NoAttentionBlock's __init__
model_no_attn.transformer.h = nn.ModuleList([NoAttentionBlock(config) for _ in range(config.n_layer)]).to(device)
optimizer = torch.optim.AdamW(model_no_attn.parameters(), lr=learning_rate)

print(f"Training GPT WITHOUT Attention ({sum(p.numel() for p in model_no_attn.parameters())/1e6:.2f}M params)...")

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model_no_attn)
        results['no_attention']['train'].append(losses['train'].item())
        results['no_attention']['val'].append(losses['val'].item())
        results['no_attention']['steps'].append(iter)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model_no_attn(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
plt.figure(figsize=(12, 7))
# Plot training and validation for both architectures
if results['with_attention']['steps']:
    plt.plot(results['with_attention']['steps'], results['with_attention']['train'], 'b-', label='With Attention (Train)')
    plt.plot(results['with_attention']['steps'], results['with_attention']['val'], 'b--', label='With Attention (Val)')
if results['no_attention']['steps']:
    plt.plot(results['no_attention']['steps'], results['no_attention']['train'], 'r-', label='No Attention (Train)')
    plt.plot(results['no_attention']['steps'], results['no_attention']['val'], 'r--', label='No Attention (Val)')

plt.title('Performance Comparison: Multi-Head Attention vs. Feed-Forward Only')
plt.xlabel('Iterations')
plt.ylabel('Cross-Entropy Loss')
plt.legend()
plt.grid(True)
plt.show()

### Line-by-Line Documentation

- `Head`: Implements one channel of self-attention. It projects inputs into Queries, Keys, and Values. The dot-product of Q and K (scaled by square root of dimension) creates the attention matrix, which is then masked to prevent looking ahead.
- `MultiHeadAttention`: Wraps multiple `Head` instances and concatenates their outputs, followed by a linear projection. This allows the model to learn multiple relationships for each token.
- `FeedForward`: A standard MLP applied to each token position independently.
- `Block`: The fundamental building block. It uses LayerNorm before the sub-layers (Pre-Norm architecture) and residual connections to keep the signal strong.
- `GPT`: The top-level container. It handles token and position embeddings, stacks the Blocks, and applies the final LayerNorm and Linear head to produce vocabulary logits.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(steps, train_losses, label='Train Loss')
plt.plot(steps, val_losses, label='Val Loss')
plt.title('Training and Validation Loss (With Attention)')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

print("Overfitting Analysis: If Val Loss starts rising while Train Loss falls, the model is overfitting.")

In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config.n_embd, config.n_head) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight
        self.apply(self._init_weights)
        print("number of parameters: %.2fM" % (sum(p.numel() for p in self.parameters())/1e6,))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        # Fix: Ensure positional tensor is created on the correct device
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

### Custom Transformer Implementation
We are now implementing our own Multi-Head Attention and Layer Normalization to replace the previous redundant class definitions.

In [ ]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
### Detailed Explanation of the Transformer Steps

1. **Token & Position Embedding**: Converts raw integer indices into high-dimensional vectors and adds spatial information.
2. **Head (Self-Attention)**: Each token emits a `query`, `key`, and `value`. It looks at other tokens' `keys` to find matches for its `query`, then weights the `values` accordingly.
3. **Causal Masking**: We use a triangular matrix (`tril`) to ensure token *i* cannot see token *i+1*.
4. **Multi-Head Attention**: Concatenates multiple 'Heads' so the model can learn different types of relationships simultaneously.
5. **FeedForward**: A sub-network that processes each token individually.
6. **Residual Connections & LayerNorm**: Allows gradients to flow through deep networks without vanishing while keeping activations stable.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

eval_iters = 200
eval_interval = 500

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            X, Y = X.to(device), Y.to(device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
max_iters = 5000
learning_rate = 3e-4

# Re-initialize optimizer for the full run
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')
    xb, yb = xb.to(device), yb.to(device)

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_output = model.generate(context, max_new_tokens=500)
print(decode(generated_output[0].tolist()))

In [ ]:
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))